In [0]:
%pip install openai

In [0]:



import mlflow
import mlflow.pyfunc
import os
import pandas as pd
class Model(mlflow.pyfunc.PythonModel):

    def context(self):
        from openai import OpenAI
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise RuntimeError("OPENAI_API_KEY not set")
        # api_key = dbutils.secrets.get(scope = "openai", key = "apikey")
        self.client = OpenAI(api_key=api_key)


    def predict(self, context, file_path = "/Volumes/voice_graph_rag/default/voice_data/recording_20251217_224657.wav"):
        
        audio_file= open(file_path, "rb")
        transcription = self.client.audio.transcriptions.create(
            model="gpt-4o-transcribe", 
            file=audio_file
        )
        return transcription.text

# with mlflow.start_run():
#     mlflow.pyfunc.log_model("audio-transcriber", 
#                             python_model=Model(), 
#                             pip_requirements=["openai==2.13.0"]
#     )

# Example input for signature inference
input_example = pd.DataFrame({"file_path": ["/Volumes/voice_graph_rag/default/voice_data/recording_20251217_224657.wav"]})
signature = mlflow.models.infer_signature(input_example, pd.DataFrame({"transcription": ["example text"]}))

with mlflow.start_run():
    mlflow.pyfunc.log_model(
        "audio-transcriber",
        python_model=Model(),
        pip_requirements=["openai==2.13.0"],
        registered_model_name="voice_graph_rag.default.audio_transcriber",
        signature=signature,
        input_example=input_example
    )

In [0]:
import requests

endpoint_config = {
    "name": "speech-to-text-endpoint",
    "config": {
        "served_entities": [
            {
                "entity_name": "voice_graph_rag.default.audio_transcriber",
                "entity_version": "1",
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
                "environment_vars": {
                    "OPENAI_API_KEY": "{{secrets/openai/apikey}}"
                }
            }
        ]
    }
}

endpoint = client.create_endpoint(name = 'speech-to-text',
                                  config = endpoint_config)